# Estimating the Effect of Proximity to "Good" Primary Schools on HDB Resale Prices

This notebook provides a full methodology and modeling pipeline for your project brief:

> Estimate the effect of being located near a "good" primary school on HDB resale prices.

It includes:
- Data integration and cleaning
- Geospatial feature engineering (distance to good schools)
- EDA and visualization
- Predictive modeling
- Causal estimation (fixed-effects hedonic regression, matching, local RDD)
- Robustness and placebo checks


## 0) Recommended datasets (beyond current repo)

You already have school-level features in `school_scoring/schools_tiered.csv`. To estimate housing price effects credibly, add:

1. **HDB resale transactions** (Data.gov.sg)  
   Fields: `month, town, block, street_name, flat_type, floor_area_sqm, lease_commence_date, resale_price, storey_range, flat_model`
2. **HDB property information** (Data.gov.sg)  
   Use for `latitude, longitude` by `block + street_name + town`.
3. **MRT station locations** (LTA/OneMap/open geodata)  
   For accessibility controls.
4. **Macro controls** (optional): CPI / mortgage rates by month  
   To avoid confounding from market cycles.

Expected local file paths used in this notebook:
- `school_scoring/schools_tiered.csv` (existing)
- `data_external/hdb_resale.csv` (add)
- `data_external/hdb_property_info.csv` (add)
- `data_external/mrt_stations.csv` (optional)


In [ ]:
# Install dependencies when needed.
# Important: run this once in a fresh environment.
%pip install -q pandas numpy matplotlib seaborn scikit-learn statsmodels scipy


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors

import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 120)


In [ ]:
# -------------------------
# Configurable file locations
# -------------------------
PATH_SCHOOLS = Path('school_scoring/schools_tiered.csv')
PATH_RESALE = Path('data_external/hdb_resale.csv')
PATH_PROP = Path('data_external/hdb_property_info.csv')
PATH_MRT = Path('data_external/mrt_stations.csv')  # optional

assert PATH_SCHOOLS.exists(), f'Missing required file: {PATH_SCHOOLS}'
print('Schools file found:', PATH_SCHOOLS)
print('Resale file exists:', PATH_RESALE.exists())
print('Property file exists:', PATH_PROP.exists())
print('MRT file exists (optional):', PATH_MRT.exists())


## 1) Load and inspect school quality data

We use your existing `school_tier` and `school_score` to define "good" schools.
Default definition in this notebook:
- **Good school = Tier 1 or Tier 2** (you can tighten to Tier 1 only in robustness checks)


In [ ]:
schools = pd.read_csv(PATH_SCHOOLS)
schools.columns = [c.strip() for c in schools.columns]

# Keep/rename fields we need for geospatial merge and quality labeling
schools['postal_code'] = schools['postal_code'].astype(str).str.zfill(6)
schools['good_school_t12'] = schools['school_tier'].isin(['Tier 1', 'Tier 2']).astype(int)
schools['good_school_t1'] = (schools['school_tier'] == 'Tier 1').astype(int)

schools[['school_name', 'postal_code', 'school_tier', 'school_score', 'good_school_t12', 'good_school_t1']].head()


## 2) Load HDB resale + property geocode data

If these files are not present yet, add them first (recommended from Data.gov.sg exports).


In [ ]:
if not PATH_RESALE.exists() or not PATH_PROP.exists():
    raise FileNotFoundError(
        'Please add data_external/hdb_resale.csv and data_external/hdb_property_info.csv before running modeling sections.'
    )

resale = pd.read_csv(PATH_RESALE)
prop = pd.read_csv(PATH_PROP)

print('Resale shape:', resale.shape)
print('Property shape:', prop.shape)
display(resale.head(2))
display(prop.head(2))


In [ ]:
# -------------------------
# Column harmonization helpers
# -------------------------
def pick_col(df, candidates, required=True):
    lower_map = {c.lower().strip(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    if required:
        raise KeyError(f'Missing expected columns. Tried: {candidates}')
    return None

# Resale columns (flexible to common naming variants)
col_month = pick_col(resale, ['month'])
col_town = pick_col(resale, ['town'])
col_block = pick_col(resale, ['block'])
col_street = pick_col(resale, ['street_name'])
col_price = pick_col(resale, ['resale_price'])
col_area = pick_col(resale, ['floor_area_sqm'])
col_lease = pick_col(resale, ['lease_commence_date'])
col_type = pick_col(resale, ['flat_type'])
col_model = pick_col(resale, ['flat_model'])
col_storey = pick_col(resale, ['storey_range'])

# Property columns
p_town = pick_col(prop, ['town'])
p_block = pick_col(prop, ['block'])
p_street = pick_col(prop, ['street_name'])
p_lat = pick_col(prop, ['latitude', 'lat'])
p_lon = pick_col(prop, ['longitude', 'lon', 'lng'])
p_postal = pick_col(prop, ['postal', 'postal_code'], required=False)

print('Column mapping successful.')


In [ ]:
# -------------------------
# Clean key fields and merge resale with geocodes
# -------------------------
for d in [resale, prop]:
    d['_town'] = d[pick_col(d, ['town'])].astype(str).str.upper().str.strip()
    d['_block'] = d[pick_col(d, ['block'])].astype(str).str.upper().str.strip()
    d['_street'] = d[pick_col(d, ['street_name'])].astype(str).str.upper().str.strip()

prop_geo = (
    prop
    .dropna(subset=[p_lat, p_lon])
    .drop_duplicates(subset=['_town', '_block', '_street'])
    [['_town', '_block', '_street', p_lat, p_lon] + ([p_postal] if p_postal else [])]
    .copy()
)
prop_geo = prop_geo.rename(columns={p_lat: 'latitude', p_lon: 'longitude'})

hdb = resale.merge(prop_geo, on=['_town', '_block', '_street'], how='left')

# Main cleaned fields
hdb['month'] = pd.to_datetime(hdb[col_month].astype(str) + '-01', errors='coerce')
hdb['resale_price'] = pd.to_numeric(hdb[col_price], errors='coerce')
hdb['floor_area_sqm'] = pd.to_numeric(hdb[col_area], errors='coerce')
hdb['lease_commence_date'] = pd.to_numeric(hdb[col_lease], errors='coerce')
hdb['flat_age'] = hdb['month'].dt.year - hdb['lease_commence_date']
hdb['town'] = hdb[col_town].astype(str).str.upper().str.strip()
hdb['flat_type'] = hdb[col_type].astype(str).str.upper().str.strip()
hdb['flat_model'] = hdb[col_model].astype(str).str.upper().str.strip()
hdb['storey_range'] = hdb[col_storey].astype(str).str.upper().str.strip()

# Keep plausible observations
hdb = hdb[(hdb['resale_price'] > 10000) & (hdb['floor_area_sqm'] > 20)].copy()
hdb = hdb.dropna(subset=['month', 'resale_price', 'latitude', 'longitude'])

hdb['log_price'] = np.log(hdb['resale_price'])

print('Merged HDB sample:', hdb.shape)
hdb[['month', 'town', 'resale_price', 'floor_area_sqm', 'flat_age', 'latitude', 'longitude']].head()


## 3) Geospatial feature engineering

Core treatment idea:
- Compute each HDB unit's distance to the nearest **good** primary school.
- Define treatment as `near_good_school_1km = 1` if nearest distance <= 1km.

This operationalizes "located near a good school" with a clear threshold.


In [ ]:
# Haversine distance in km
# Important: robust vectorized approach so this can scale to larger transaction datasets.
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c


In [ ]:
# We need school coordinates. Two pathways:
# 1) If your property info has postal with lat/lon, map school postal_code -> coordinates.
# 2) Else provide a school geocode file and merge externally.

if p_postal and p_postal in prop.columns:
    school_geo_map = (
        prop.dropna(subset=[p_postal, p_lat, p_lon])
            .assign(_postal=lambda d: d[p_postal].astype(str).str.zfill(6))
            .drop_duplicates(subset=['_postal'])
            [['_postal', p_lat, p_lon]]
            .rename(columns={'_postal': 'postal_code', p_lat: 'school_lat', p_lon: 'school_lon'})
    )
    schools_geo = schools.merge(school_geo_map, on='postal_code', how='left')
else:
    schools_geo = schools.copy()
    schools_geo['school_lat'] = np.nan
    schools_geo['school_lon'] = np.nan

if schools_geo[['school_lat', 'school_lon']].isna().all().all():
    raise ValueError(
        'School coordinates unavailable. Add school_lat/school_lon columns to schools_tiered.csv '
        'or provide postal geocode mapping through property dataset.'
    )

schools_geo = schools_geo.dropna(subset=['school_lat', 'school_lon']).copy()
print('Schools with coordinates:', schools_geo.shape[0])


In [ ]:
# Distance to nearest good school (Tier 1 or Tier 2)
good_sch = schools_geo[schools_geo['good_school_t12'] == 1].copy()

# Compute nearest distance by chunking for memory safety on large data
hdb_coords = hdb[['latitude', 'longitude']].to_numpy()
sch_coords = good_sch[['school_lat', 'school_lon']].to_numpy()

nearest_dist = np.full(len(hdb), np.nan)
nearest_school = np.array([''] * len(hdb), dtype=object)

chunk = 2000
for i in range(0, len(hdb), chunk):
    j = min(i + chunk, len(hdb))
    a = hdb_coords[i:j]

    # Broadcast distances: shape (chunk_size, num_schools)
    d = haversine_km(
        a[:, None, 0], a[:, None, 1],
        sch_coords[None, :, 0], sch_coords[None, :, 1]
    )
    idx = d.argmin(axis=1)
    nearest_dist[i:j] = d[np.arange(d.shape[0]), idx]
    nearest_school[i:j] = good_sch.iloc[idx]['school_name'].values

hdb['dist_to_good_school_km'] = nearest_dist
hdb['nearest_good_school_name'] = nearest_school
hdb['near_good_school_1km'] = (hdb['dist_to_good_school_km'] <= 1.0).astype(int)
hdb['near_good_school_500m'] = (hdb['dist_to_good_school_km'] <= 0.5).astype(int)
hdb['near_good_school_1500m'] = (hdb['dist_to_good_school_km'] <= 1.5).astype(int)

hdb[['dist_to_good_school_km', 'near_good_school_1km']].describe()


## 4) EDA: treatment prevalence and raw price patterns


In [ ]:
display(hdb['near_good_school_1km'].value_counts().rename('count').to_frame())

ax = sns.countplot(data=hdb, x='near_good_school_1km', palette='Set2')
ax.set_title('Treatment Group Sizes: Near Good School (<=1km)')
ax.set_xlabel('near_good_school_1km')
ax.set_ylabel('Count')
plt.show()


In [ ]:
ax = sns.boxplot(data=hdb.sample(min(len(hdb), 50000), random_state=42),
                 x='near_good_school_1km', y='resale_price', palette='coolwarm')
ax.set_title('Raw Resale Price by School Proximity Treatment')
ax.set_xlabel('near_good_school_1km')
ax.set_ylabel('resale_price')
plt.yscale('log')
plt.show()


In [ ]:
# Monthly trend comparison
trend = (
    hdb.groupby(['month', 'near_good_school_1km'])['resale_price']
    .median()
    .reset_index()
)

plt.figure(figsize=(12, 6))
for k, g in trend.groupby('near_good_school_1km'):
    label = 'Near good school (<=1km)' if k == 1 else 'Not near good school (>1km)'
    plt.plot(g['month'], g['resale_price'], label=label)

plt.title('Median Monthly Resale Price by Treatment Group')
plt.xlabel('Month')
plt.ylabel('Median resale_price')
plt.legend()
plt.tight_layout()
plt.show()


## 5) Predictive models (price forecasting benchmark)

These models are not causal by themselves, but they check whether school proximity adds predictive signal.

Models:
- Linear Regression
- Random Forest
- HistGradientBoosting

Evaluation:
- Time-based split to reduce leakage
- MAE, RMSE, R² on holdout set


In [ ]:
# Build modeling table
model_df = hdb.copy()
model_df['month_str'] = model_df['month'].dt.to_period('M').astype(str)

feature_num = ['floor_area_sqm', 'flat_age', 'dist_to_good_school_km']
feature_cat = ['town', 'flat_type', 'flat_model', 'storey_range', 'month_str']
feature_bin = ['near_good_school_1km']
X_cols = feature_num + feature_cat + feature_bin

y = model_df['log_price']
X = model_df[X_cols]

# Time split at 80% quantile month
cutoff = model_df['month'].quantile(0.80)
train_idx = model_df['month'] <= cutoff
test_idx = ~train_idx

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print('Train size:', len(X_train), 'Test size:', len(X_test), 'Cutoff:', cutoff.date())


In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, feature_num + feature_bin),
    ('cat', categorical_pipe, feature_cat)
])

models = {
    'linear_regression': LinearRegression(),
    'random_forest': RandomForestRegressor(
        n_estimators=250,
        max_depth=18,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
    'hist_gradient_boosting': HistGradientBoostingRegressor(
        max_depth=8,
        learning_rate=0.05,
        max_iter=400,
        random_state=42
    )
}

results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([
        ('prep', preprocess),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    pred_log = pipe.predict(X_test)

    # Evaluate in original price scale for business interpretability
    pred_price = np.exp(pred_log)
    true_price = np.exp(y_test)

    mae = mean_absolute_error(true_price, pred_price)
    rmse = mean_squared_error(true_price, pred_price, squared=False)
    r2 = r2_score(true_price, pred_price)

    results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    fitted[name] = pipe

pd.DataFrame(results).sort_values('RMSE')


## 6) Causal model A: Hedonic regression with fixed effects

Main estimating equation:

`log_price ~ near_good_school_1km + housing_controls + C(town) + C(month)`

Interpretation of treatment coefficient:
- Approximate percentage effect = `100 * (exp(beta) - 1)`


In [ ]:
causal_df = hdb.copy()
causal_df['month_str'] = causal_df['month'].dt.to_period('M').astype(str)

# Keep columns used in formula and drop NA
causal_df = causal_df.dropna(subset=['log_price', 'near_good_school_1km', 'floor_area_sqm', 'flat_age', 'town', 'month_str'])

formula = (
    'log_price ~ near_good_school_1km + floor_area_sqm + flat_age '
    '+ C(flat_type) + C(flat_model) + C(storey_range) + C(town) + C(month_str)'
)

fe_model = smf.ols(formula=formula, data=causal_df).fit(cov_type='HC1')
print(fe_model.summary().tables[1])

beta = fe_model.params.get('near_good_school_1km', np.nan)
if pd.notna(beta):
    pct = 100 * (np.exp(beta) - 1)
    print(f'Estimated proximity effect (FE OLS): {pct:.2f}%')


## 7) Causal model B: Propensity score matching (PSM)

Goal:
- Compare treated flats with observationally similar untreated flats.

Steps:
1. Estimate treatment propensity.
2. Match each treated flat to nearest untreated by propensity score.
3. Compute ATT on log price and resale price.


In [ ]:
psm_df = hdb.copy().dropna(subset=['near_good_school_1km', 'log_price', 'floor_area_sqm', 'flat_age'])
psm_df['month_str'] = psm_df['month'].dt.to_period('M').astype(str)

psm_features_num = ['floor_area_sqm', 'flat_age']
psm_features_cat = ['town', 'flat_type', 'flat_model', 'storey_range', 'month_str']

X_psm = psm_df[psm_features_num + psm_features_cat]
T = psm_df['near_good_school_1km'].astype(int)

psm_prep = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler())
    ]), psm_features_num),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oh', OneHotEncoder(handle_unknown='ignore'))
    ]), psm_features_cat)
])

X_psm_tr = psm_prep.fit_transform(X_psm)
prop_model = LogisticRegression(max_iter=500)
prop_model.fit(X_psm_tr, T)

psm_df['propensity'] = prop_model.predict_proba(X_psm_tr)[:, 1]

treated = psm_df[psm_df['near_good_school_1km'] == 1].copy()
control = psm_df[psm_df['near_good_school_1km'] == 0].copy()

nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[['propensity']])
dist, idx = nn.kneighbors(treated[['propensity']])
matched_control = control.iloc[idx.flatten()].copy()

att_log = (treated['log_price'].values - matched_control['log_price'].values).mean()
att_pct = 100 * (np.exp(att_log) - 1)
att_price = (treated['resale_price'].values - matched_control['resale_price'].values).mean()

print('Treated count:', len(treated), 'Matched control count:', len(matched_control))
print(f'ATT (log price): {att_log:.4f} -> {att_pct:.2f}%')
print(f'ATT (resale price, SGD): {att_price:,.2f}')


In [ ]:
# Balance diagnostics (standardized mean differences) on key numeric covariates

def smd(a, b):
    va = np.var(a, ddof=1)
    vb = np.var(b, ddof=1)
    pooled = np.sqrt((va + vb) / 2)
    return 0 if pooled == 0 else (np.mean(a) - np.mean(b)) / pooled

before = {
    c: smd(psm_df.loc[psm_df['near_good_school_1km'] == 1, c],
           psm_df.loc[psm_df['near_good_school_1km'] == 0, c])
    for c in ['floor_area_sqm', 'flat_age']
}

after = {
    c: smd(treated[c], matched_control[c])
    for c in ['floor_area_sqm', 'flat_age']
}

balance = pd.DataFrame({'before_match_smd': before, 'after_match_smd': after})
display(balance)


## 8) Causal model C: Local RDD around 1km cutoff

Design intuition:
- Compare flats just inside vs just outside the 1km boundary.
- Restrict sample to a bandwidth around 1km to increase local comparability.

This is not a perfect RDD (location sorting can exist), but it is a strong robustness design.


In [ ]:
rdd = hdb.copy().dropna(subset=['dist_to_good_school_km', 'log_price'])

# Bandwidth around 1km cutoff (adjustable)
bw = 0.5
rdd = rdd[(rdd['dist_to_good_school_km'] >= 1.0 - bw) & (rdd['dist_to_good_school_km'] <= 1.0 + bw)].copy()

rdd['running'] = rdd['dist_to_good_school_km'] - 1.0
rdd['treat'] = (rdd['running'] <= 0).astype(int)
rdd['month_str'] = rdd['month'].dt.to_period('M').astype(str)

# Local linear with interaction allows slope change across cutoff
rdd_formula = (
    'log_price ~ treat + running + treat:running + floor_area_sqm + flat_age '
    '+ C(flat_type) + C(town) + C(month_str)'
)

rdd_model = smf.ols(rdd_formula, data=rdd).fit(cov_type='HC1')
print(rdd_model.summary().tables[1])

beta_rdd = rdd_model.params.get('treat', np.nan)
if pd.notna(beta_rdd):
    pct_rdd = 100 * (np.exp(beta_rdd) - 1)
    print(f'Local discontinuity estimate at 1km cutoff: {pct_rdd:.2f}%')


In [ ]:
# Visual RDD diagnostic
sample_rdd = rdd.sample(min(len(rdd), 60000), random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample_rdd, x='dist_to_good_school_km', y='resale_price', alpha=0.15, s=12)
plt.axvline(1.0, color='red', linestyle='--', label='1km cutoff')
plt.yscale('log')
plt.title('RDD Diagnostic: Resale Price vs Distance to Nearest Good School')
plt.xlabel('Distance to nearest good school (km)')
plt.ylabel('Resale price (log scale)')
plt.legend()
plt.tight_layout()
plt.show()


## 9) Robustness checks

We test whether results are sensitive to treatment definition.


In [ ]:
robust_rows = []

for tr_col in ['near_good_school_500m', 'near_good_school_1km', 'near_good_school_1500m']:
    tmp = hdb.copy().dropna(subset=['log_price', tr_col, 'floor_area_sqm', 'flat_age'])
    tmp['month_str'] = tmp['month'].dt.to_period('M').astype(str)

    f = (
        f'log_price ~ {tr_col} + floor_area_sqm + flat_age '
        '+ C(flat_type) + C(flat_model) + C(storey_range) + C(town) + C(month_str)'
    )
    m = smf.ols(f, data=tmp).fit(cov_type='HC1')
    b = m.params.get(tr_col, np.nan)
    se = m.bse.get(tr_col, np.nan)
    robust_rows.append({
        'treatment_def': tr_col,
        'coef_log': b,
        'std_err': se,
        'effect_pct': 100 * (np.exp(b) - 1) if pd.notna(b) else np.nan
    })

robust_df = pd.DataFrame(robust_rows)
display(robust_df)

ax = sns.barplot(data=robust_df, x='treatment_def', y='effect_pct', palette='mako')
ax.set_title('Robustness: Estimated Effect Across Distance Thresholds')
ax.set_xlabel('Treatment definition')
ax.set_ylabel('Estimated effect (%)')
plt.tight_layout()
plt.show()


## 10) Placebo test

Placebo idea:
- Replace treatment with proximity to **non-good schools only**.
- If placebo effect is similar in size, your main estimate may still contain location confounding.


In [ ]:
non_good = schools_geo[schools_geo['good_school_t12'] == 0].copy()

if len(non_good) > 0:
    hdb_coords = hdb[['latitude', 'longitude']].to_numpy()
    sch_coords = non_good[['school_lat', 'school_lon']].to_numpy()
    nearest_placebo = np.full(len(hdb), np.nan)

    chunk = 2000
    for i in range(0, len(hdb), chunk):
        j = min(i + chunk, len(hdb))
        a = hdb_coords[i:j]
        d = haversine_km(
            a[:, None, 0], a[:, None, 1],
            sch_coords[None, :, 0], sch_coords[None, :, 1]
        )
        nearest_placebo[i:j] = d.min(axis=1)

    hdb['near_placebo_school_1km'] = (nearest_placebo <= 1.0).astype(int)

    tmp = hdb.copy().dropna(subset=['log_price', 'near_placebo_school_1km', 'floor_area_sqm', 'flat_age'])
    tmp['month_str'] = tmp['month'].dt.to_period('M').astype(str)

    placebo_model = smf.ols(
        'log_price ~ near_placebo_school_1km + floor_area_sqm + flat_age + C(flat_type) + C(town) + C(month_str)',
        data=tmp
    ).fit(cov_type='HC1')

    b = placebo_model.params.get('near_placebo_school_1km', np.nan)
    print(placebo_model.summary().tables[1])
    print(f'Placebo effect (%): {100 * (np.exp(b) - 1):.2f}%')
else:
    print('No non-good schools available for placebo test.')


## 11) Model comparison summary


In [ ]:
summary_rows = []

# FE OLS main estimate
if 'fe_model' in globals() and 'near_good_school_1km' in fe_model.params.index:
    b = fe_model.params['near_good_school_1km']
    summary_rows.append({
        'method': 'FE OLS (hedonic)',
        'target_effect_pct': 100 * (np.exp(b) - 1)
    })

# PSM ATT
if 'att_pct' in globals():
    summary_rows.append({'method': 'PSM ATT', 'target_effect_pct': att_pct})

# RDD
if 'rdd_model' in globals() and 'treat' in rdd_model.params.index:
    b = rdd_model.params['treat']
    summary_rows.append({'method': 'Local RDD (1km cutoff)', 'target_effect_pct': 100 * (np.exp(b) - 1)})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if len(summary_df) > 0:
    ax = sns.barplot(data=summary_df, x='method', y='target_effect_pct', palette='rocket')
    ax.set_title('Estimated Price Premium from Proximity to Good School')
    ax.set_xlabel('Method')
    ax.set_ylabel('Estimated effect (%)')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()


## 12) Evaluation strategy and reporting checklist

For final report quality, include:
- **Primary estimate**: FE OLS coefficient + robust SE + confidence interval.
- **Triangulation**: compare FE OLS, PSM, and RDD estimates.
- **Robustness**: threshold sensitivity (500m/1km/1.5km), placebo test.
- **Predictive sanity**: out-of-time model performance (RMSE, MAE, R²).
- **Threats to validity**: omitted amenities, school catchment self-selection, unobserved block quality.

Optional upgrades:
1. Add MRT accessibility and CBD distance controls.
2. Deflate prices with CPI for real price effects.
3. Run heterogeneity by flat type / town / market regime.
4. Use spatial fixed effects (planning area or nearest station bins).
